# Transformers NSFW Nano

In [5]:
import os
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import torch

# Define local directory to save the model
model_name = "viddexa/nsfw-detection-nano"
local_model_path = "./nsfw_detection_nano"

# Download and save model and processor locally
print(f"Downloading model to {local_model_path}...")
processor = AutoImageProcessor.from_pretrained(model_name, use_fast=False)
model = AutoModelForImageClassification.from_pretrained(model_name)

# Save to local directory
processor.save_pretrained(local_model_path)
model.save_pretrained(local_model_path)
print(f"Model saved successfully to {local_model_path}")

# Now load from local directory
print("\nLoading model from local directory...")
processor = AutoImageProcessor.from_pretrained(local_model_path, use_fast=False)
model = AutoModelForImageClassification.from_pretrained(local_model_path)
model.eval()

# Get image files
image_files = [f for f in os.listdir('./test_images') if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"\n{'Image':<30} | {'Prediction':<10} | {'Confidence':>10} | Details")
print("-" * 90)

for img_file in image_files:
    img = Image.open('./test_images/' + img_file).convert("RGB")
    
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt")
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    
    pred_id = int(probs.argmax())
    label = model.config.id2label[pred_id]
    
    # Format all probabilities
    details = " | ".join([f"{model.config.id2label[i]}: {p:.2%}" for i, p in enumerate(probs)])
    
    print(f"{img_file:<30} | {label:<10} | {probs[pred_id]:>9.2%} | {details}")

Model saved successfully to ./nsfw_detection_nano

Loading model from local directory...

Image                          | Prediction | Confidence | Details
------------------------------------------------------------------------------------------
normal_2.jpg                   | safe       |   100.00% | safe: 100.00% | nsfw: 0.00%
normal_1.jpg                   | safe       |    99.98% | safe: 99.98% | nsfw: 0.02%
nsfw_high.jpg                  | nsfw       |   100.00% | safe: 0.00% | nsfw: 100.00%
nsfw_medium.jpg                | nsfw       |   100.00% | safe: 0.00% | nsfw: 100.00%


In [5]:
"""
PyTorch to ONNX Conversion Script for NSFW Detection Model
This script converts the Hugging Face PyTorch model to ONNX format with verification.
"""

import os
import torch
import onnx
import onnxruntime as ort
import numpy as np
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image


def convert_to_onnx(
    model_path: str = "./nsfw_detection_nano",
    output_path: str = "./nsfw_detection_nano.onnx",
    opset_version: int = 18,  # Use opset 18 to avoid conversion issues
    test_image_path: str = "./test_images/normal_1.jpg"
):
    """
    Convert a Hugging Face PyTorch model to ONNX format.
    
    Args:
        model_path: Path to the saved PyTorch model
        output_path: Path where ONNX model will be saved
        opset_version: ONNX opset version (default: 14)
        test_image_path: Path to test image for verification
    """
    
    print("=" * 80)
    print("PyTorch to ONNX Conversion")
    print("=" * 80)
    
    # Step 1: Load the model and processor
    print("\n[1/5] Loading PyTorch model and processor...")
    processor = AutoImageProcessor.from_pretrained(model_path, use_fast=False)
    model = AutoModelForImageClassification.from_pretrained(model_path)
    model.eval()
    model.set_swish(memory_efficient=False)
    print("✓ Model loaded successfully")
    
    # Step 2: Create dummy input for export
    print("\n[2/5] Creating dummy input tensor...")
    # Get expected input size from processor
    input_size = processor.size.get('height', 224)  # Default to 224 if not specified
    batch_size = 1
    channels = 3
    
    dummy_input = torch.randn(batch_size, channels, input_size, input_size)
    print(f"✓ Dummy input shape: {dummy_input.shape}")
    
    # Step 3: Export to ONNX
    print("\n[3/5] Exporting model to ONNX format...")
    # Use legacy exporter for better compatibility with transformers models
    with torch.no_grad():
        torch.onnx.export(
            model,
            dummy_input,
            output_path,
            export_params=True,
            opset_version=opset_version,
            do_constant_folding=True,  # Optimize constant operations
            input_names=['pixel_values'],
            output_names=['logits'],
            dynamo=False  # Use legacy exporter for better accuracy
        )
    print(f"✓ Model exported to: {output_path}")
    
    # Step 4: Verify ONNX model
    print("\n[4/5] Verifying ONNX model...")
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    print("✓ ONNX model is valid")
    
    # Step 5: Test ONNX inference and compare with PyTorch
    print("\n[5/5] Testing ONNX inference...")
    ort_session = ort.InferenceSession(output_path)
    
    # Load a test image
    if os.path.exists(test_image_path):
        test_image = Image.open(test_image_path).convert("RGB")
        
        # Prepare input for both models
        inputs = processor(images=test_image, return_tensors="pt")
        pixel_values = inputs['pixel_values'].numpy()
        
        # PyTorch inference
        with torch.no_grad():
            pytorch_output = model(**inputs).logits
            pytorch_probs = torch.softmax(pytorch_output, dim=-1)[0]
        
        # ONNX inference
        onnx_output = ort_session.run(
            ['logits'],
            {'pixel_values': pixel_values}
        )[0]
        onnx_probs = torch.softmax(torch.tensor(onnx_output), dim=-1)[0]
        
        # Compare outputs
        max_diff = torch.max(torch.abs(pytorch_probs - onnx_probs)).item()
        print(f"✓ ONNX inference successful")
        print(f"  Max difference between PyTorch and ONNX: {max_diff:.6f}")
        
        if max_diff < 1e-5:
            print("  ✓ Outputs match perfectly!")
        elif max_diff < 1e-3:
            print("  ✓ Outputs are very close (acceptable difference)")
        else:
            print("  ⚠ Warning: Outputs differ significantly")
        
        # Display predictions
        print("\nPrediction Comparison:")
        print("-" * 60)
        for i, (pytorch_prob, onnx_prob) in enumerate(zip(pytorch_probs, onnx_probs)):
            label = model.config.id2label[i]
            print(f"{label:10} | PyTorch: {pytorch_prob:.4f} | ONNX: {onnx_prob:.4f}")
    else:
        print(f"⚠ Test image not found at {test_image_path}, skipping comparison")
    
    # Summary
    print("\n" + "=" * 80)
    print("Conversion Complete!")
    print("=" * 80)
    print(f"\nONNX model saved at: {output_path}")
    if os.path.exists(output_path):
        print(f"Model size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
    
    return output_path

In [6]:
onnx_path = convert_to_onnx()
print(f"\n✓ Conversion successful! ONNX model: {onnx_path}")

PyTorch to ONNX Conversion

[1/5] Loading PyTorch model and processor...


AttributeError: 'EfficientNetForImageClassification' object has no attribute 'set_swish'